In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else 'cpu'

## Hyperparameters

The current set of hyperparameters for our model implementation is listed below:

In [2]:
# new hyperparameters
num_heads = 4

# hyperparameters
batch_size = 32              # how many independent sequences to parallel-process?
max_context_size = 8         # "block_size" ... maximum context length for predictions
max_iters = 5000
eval_interval = 500
learning_rate = 1e-3
loss_estimation_iters = 200  # "eval_iters" ... num. iters for estimate_loss
embedding_size = 32          # "n_embd" ... size of the embedding tensors
head_size = 16               # size of a single head of attention

----

## "Magical" Helper

In [3]:
from helper import *

def get_batch(split):
    data = training_data if split == 'train' else validation_data
    ix = torch.randint(len(data) - max_context_size, (batch_size,))
    x = torch.stack([data[i:i+max_context_size] for i in ix]).to(device)
    y = torch.stack([data[i+1:i+max_context_size+1] for i in ix]).to(device)
    return x,y

----

## Multi-Head Attention



In [4]:
class Head(nn.Module):
    """ one head of self-attention """
    def __init__(self, head_size):
        super().__init__()

        self.key = nn.Linear(embedding_size, head_size, bias=False)
        self.query = nn.Linear(embedding_size, head_size, bias=False)
        self.value = nn.Linear(embedding_size, head_size, bias=False)

        self.register_buffer(
            'tril',
            torch.tril(torch.ones(max_context_size, max_context_size))
        )

    def forward(self, x):
        B,T,C = x.shape

        k = self.key(x)      # (B,T,C)
        q = self.query(x)    # (B,T,C)

        wei = q @ k.transpose(-2,-1) * C**-0.5   # (B,T,C) @ (B,C,T) ----> (B,T,T)
        wei = wei.masked_fill(self.tril[:T, :T] == 0, float('-inf'))     # (B,T,T)
        wei = F.softmax(wei, dim=-1)                                     # (B,T,T)
        v = self.value(x)    # (B,T,C)
        out = wei @ v        # (B,T,T) @ (B,T,C) ----> (B,T,C)

        return out

In [5]:
class MultiHeadAttention(nn.Module):
    """ multiple heads of self-attention in parallel """

    def __init__(self, num_heads, head_size):
        super().__init__()
        self.heads = nn.ModuleList([Head(head_size) for _ in range(num_heads)])

    def forward(self, x):
        # simply concatenate the head outputs in the Channel dimension
        return torch.cat([h(x) for h in self.heads], dim=-1)

In [6]:
# bigram language model with multiple-head self-attention
class BigramLanguageModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.token_embedding_table = nn.Embedding(vocab_size, embedding_size)
        self.position_embedding_table = nn.Embedding(max_context_size, embedding_size)

        # 4 heads of 8-dimensional self-attention
        self.sa_heads = MultiHeadAttention(num_heads, embedding_size//num_heads)

        self.lm_head = nn.Linear(embedding_size, vocab_size)

    def forward(self, idx, targets=None):
        B,T = idx.shape

        # idx and targets are both (B,T) tensor of int
        tok_embeddings = self.token_embedding_table(idx)  # (B,T,C)
        pos_embeddings = self.position_embedding_table(torch.arange(T, device=device))  # (T,C)

        x = tok_embeddings + pos_embeddings  # (B,T,C)
        x = self.sa_heads(x)                 # (B,T,C)
        logits = self.lm_head(x)             # (B,T,vocab_size)

        if targets is None:
            loss = None
        else:
            B,T,C = logits.shape
            logits = logits.view(B*T, C)
            targets = targets.view(B*T)
            loss = F.cross_entropy(logits, targets)

        return logits, loss

    def generate(self, idx, max_new_tokens):
        # idx is (B,T) tensor of indices in the current context
        for _ in range(max_new_tokens):

            # crop idx to the last max_context_size tokens
            idx_cond = idx[:, -max_context_size:]

            logits, loss = self(idx_cond)
            logits = logits[:, -1, :]           # becomes (B,C)
            probs = F.softmax(logits, dim=-1)   # (B,C)
            idx_next = torch.multinomial(probs, num_samples=1) # (B,1)
            idx = torch.cat((idx, idx_next), dim=1)  # (B,T+1)
        return idx

## The Training Loop

In [7]:
%%time

torch.manual_seed(1337)

model = BigramLanguageModel().to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(loss_estimation_iters)
        for k in range(loss_estimation_iters):
            X,Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

for iter in range(max_iters):

    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step {iter}: train loss is {losses['train']:.4f}, val loss is {losses['val']:.4f}")

    xb,yb = get_batch('train')
    logits, loss = model(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(f"final loss: {loss.item():.4f}\n")

step 0: train loss is 4.2227, val loss is 4.2226
step 500: train loss is 2.6592, val loss is 2.6733
step 1000: train loss is 2.4980, val loss is 2.5064
step 1500: train loss is 2.4291, val loss is 2.4349
step 2000: train loss is 2.3716, val loss is 2.3844
step 2500: train loss is 2.3417, val loss is 2.3561
step 3000: train loss is 2.3149, val loss is 2.3347
step 3500: train loss is 2.2918, val loss is 2.3171
step 4000: train loss is 2.2895, val loss is 2.2868
step 4500: train loss is 2.2748, val loss is 2.2858
final loss: 2.4208

CPU times: user 22.1 s, sys: 391 ms, total: 22.5 s
Wall time: 23.1 s


In [8]:
# generate from the model
context = torch.zeros((1, 1), dtype=torch.int, device=device)
print(decode(
    model.generate(
        context,
        max_new_tokens=500
    )[0].tolist()
))


Whent if bridcowd, whis byer that set bobe toe anthr-and mealleands:
Warth foulque, vet?
Wedtlay anes wice my.

HDY'n om oroug
Yowns, tof is heir thil; dill, aes isee sen cin lat Hetilrov the and Win now onderabousel.

SFAUS:
Shenser cechiry prugh aissthe, ye wing, u not
To thig I whomeny wod mothake ont---An hat evibys wietit, stile weeshirecs poor gier; to
To k danteref If sor; igre! mef thre inledo the af Pre?

WISo myay I sup!
Atied is:
Sadsal the E'd st hoin couk aar tey Iry to I frouf voul


----

## Count the Model Weights

How many parameters does this model implementation have?

In [9]:
c = 0
for name, p in model.named_parameters():
    print(f"{name:60s} {p.numel():>10,}")
    c += p.numel()

print(f"{''.join(['-']*71)}")
print(f"{'total parameters':60s} {c:>10,}")

token_embedding_table.weight                                      2,080
position_embedding_table.weight                                     256
sa_heads.heads.0.key.weight                                         256
sa_heads.heads.0.query.weight                                       256
sa_heads.heads.0.value.weight                                       256
sa_heads.heads.1.key.weight                                         256
sa_heads.heads.1.query.weight                                       256
sa_heads.heads.1.value.weight                                       256
sa_heads.heads.2.key.weight                                         256
sa_heads.heads.2.query.weight                                       256
sa_heads.heads.2.value.weight                                       256
sa_heads.heads.3.key.weight                                         256
sa_heads.heads.3.query.weight                                       256
sa_heads.heads.3.value.weight                                   